## Task 1: Data Preparation

We load the customer dataset and scale all features using StandardScaler before applying K-Means.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/q2_customers.csv')

print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
df.head()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
X_scaled_df = pd.DataFrame(X_scaled, columns=df.columns)

print("Scaled data sample:")
X_scaled_df.head()

**Why scaling is essential before K-Means:**

K-Means assigns cluster membership by computing Euclidean distances between data points and centroids. In this dataset, `annual_spend` ranges from 5,038 to 119,757 while `visits_per_month` ranges from roughly 1 to 20 and `num_categories_purchased` from 1 to 9. Without scaling, the distance calculation is completely dominated by `annual_spend` — a difference of 10,000 in spend dwarfs any difference in visit frequency, making those features effectively invisible to the algorithm. StandardScaler transforms every feature to have mean 0 and standard deviation 1, ensuring that all six features contribute equally to the distance computation regardless of their original units or magnitude.

## Task 2: Choosing K — Elbow Method

We compute the Within-Cluster Sum of Squares (WCSS) for K = 1 through 10 and plot the elbow curve to identify the optimal number of clusters.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, wcss, marker='o', linewidth=2, color='steelblue')
plt.title('Elbow Method — WCSS vs Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(K_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Optimal K selection:**

The WCSS values are: K=1: 3000.00, K=2: 969.00, K=3: 561.25, K=4: 444.93, K=5: 402.37. The steepest drops occur at K=2 and K=3. From K=3 to K=4 the reduction is 116 points (20.7%), while from K=4 to K=5 it is only 43 points (9.6%) — a significant flattening. The elbow point therefore falls at **K = 4**, where the marginal gain from adding another cluster becomes substantially smaller. Beyond K=4, additional clusters produce diminishing reductions in WCSS, indicating that they split existing groups rather than discovering meaningfully new ones. We proceed with K = 4.

## Task 3: K-Means Clustering

We fit K-Means with K = 4, assign cluster labels, print the centroids in original feature space, and interpret each cluster in business terms.

In [ ]:
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print("Cluster counts:\n", df['cluster'].value_counts().sort_index())

In [ ]:
centroids = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=df.columns[:-1]
)
centroids.index.name = 'Cluster'
print("Cluster Centroids (original feature scale):")
centroids.round(2)

**Cluster interpretation (business terms):**

Based on the centroid values produced above, the four clusters represent the following customer segments:

- **Cluster 0 — Young frequent low-spenders (n=170):** Average age 25, annual spend £14,847, 14 visits/month, basket size £559, only 9 days since last visit, purchasing across just 2 categories. These are young, highly engaged customers who visit often but spend little per trip and explore only a narrow product range. The priority strategy is upselling and cross-category promotion to increase basket size and category breadth.

- **Cluster 1 — Lapsed high-value customers (n=80):** Average age 57, annual spend £89,814, only 2.5 visits/month, basket size £5,296, 148 days since last visit, purchasing across 7–8 categories. These customers were once high spenders across many categories but have become largely inactive — nearly 5 months since their last visit. They carry significant re-engagement value and should receive targeted win-back campaigns with strong personalised incentives.

- **Cluster 2 — Core mid-value customers (n=165):** Average age 40, annual spend £43,341, 8 visits/month, basket size £2,022, 35 days since last visit, 4–5 categories. This is the largest and most stable segment — moderately active, moderate spenders. Standard loyalty rewards and seasonal promotions should maintain their engagement.

- **Cluster 3 — Active high-value customers (n=85):** Average age 57, annual spend £89,036, 2.6 visits/month, basket size £5,751, 65 days since last visit, 7–8 categories. Similar spend and category breadth to Cluster 1 but more recently active. These are the retailer's most valuable current customers. Retention programmes, exclusive previews, and premium loyalty benefits are appropriate to sustain their engagement.

## Task 4: Dimensionality Reduction with PCA

We reduce the scaled data to 2 principal components, examine the explained variance, and interpret what each component captures based on feature loadings.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.4f} ({var*100:.2f}%)")

print(f"\nTotal variance explained: {pca.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
loadings = pd.DataFrame(
    pca.components_,
    columns=df.columns[:-1],
    index=['PC1', 'PC2']
)
print("Feature Loadings:")
loadings.round(3)

**Interpretation of principal components:**

**PC1** explains 83.56% of total variance and is a near-equal positive combination of all six features — `age` (0.412), `annual_spend` (0.422), `visits_per_month` (0.389), `basket_size` (0.420), `days_since_last_visit` (0.379), and `num_categories_purchased` (0.414). Because all loadings are positive and similar in magnitude, PC1 essentially captures overall customer activity level and lifetime value. A high PC1 score means a customer is older, spends more, visits more, has a larger basket, has been seen recently (or long ago), and purchases across many categories. It is the dominant axis separating high-engagement from low-engagement customers.

**PC2** explains only 5.57% of variance and is dominated by a very strong positive loading on `days_since_last_visit` (0.911), with negative loadings on `age` (-0.259) and `num_categories_purchased` (-0.140). PC2 therefore captures recency — specifically, whether a customer has been absent for a long time. A high PC2 score means a long-lapsed customer; a low score means a recently active one, regardless of their overall spend level.

Together the two components explain 89.13% of total variance, making this 2D PCA projection a high-fidelity representation of the original six-dimensional space.

## Task 5: Cluster Visualisation

We plot the data in PCA space with points coloured by cluster label to visually assess cluster separation.

In [ ]:
import matplotlib.pyplot as plt

colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
fig, ax = plt.subplots(figsize=(8, 6))

for cluster_id in sorted(df['cluster'].unique()):
    mask = df['cluster'] == cluster_id
    ax.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        c=colors[cluster_id],
        label=f'Cluster {cluster_id}',
        alpha=0.6,
        s=40,
        edgecolors='white',
        linewidths=0.3
    )

ax.set_title('Customer Segments — PCA Projection (PC1 vs PC2)')
ax.set_xlabel('PC1 — Overall Customer Activity and Value')
ax.set_ylabel('PC2 — Recency (higher = longer lapsed)')
ax.legend(title='Cluster')
plt.tight_layout()
plt.show()